In [ ]:
import asyncio
import time
import os
import json
import hashlib
from typing import List, Dict
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy
from bs4 import BeautifulSoup
from firecrawl import FirecrawlApp  # Assuming the SDK is named this way
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Thresholds
CATEGORY_THRESHOLDS = {
	"ABOUT_US": 200,
	"EBOOK": 200,
	"COURSES": 300,
	"RECENT_BLOG": 450,
	"TESTIMONIALS": 100,
	"WEBINAR": 150,
	"SERVICES": 150,
	"PODCAST": 200,
	"SHOP": 100,
}

# Keyword filters
CATEGORY_KEYWORDS = {
	"ABOUT_US": ["about", "who-we-are", "company", "our-story"],
	"EBOOK": ["ebook", "e-book", "downloads", "whitepaper"],
	"COURSES": ["course", "academy", "learning"],
	"RECENT_BLOG": ["blog", "insights", "articles"],
	"TESTIMONIALS": ["testimonial", "reviews", "case-study"],
	"WEBINAR": ["webinar", "event", "session"],
	"SERVICES": ["service", "solution", "capability"],
	"PODCAST": ["podcast", "listen", "episodes"],
	"SHOP": ["shop", "store", "buy"]
}

COLUMN_TO_PROCESS = "SHOP"
COLUMN_TO_READ_URL_FROM = "G"
COLUMN_TO_WRITE_URL_TO = {
	"ABOUT_US": "M",
	"EBOOK": "N",
	"COURSES": "O",
	"RECENT_BLOG": "P",
	"TESTIMONIALS": "Q",
	"WEBINAR": "R",
	"SERVICES": "S",
	"PODCAST": "T",
	"SHOP": "U"
}

CACHE_DIR = "firecrawl_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

class FirecrawlWrapper:
	def __init__(self, api_key):
		self.app = FirecrawlApp(api_key=api_key)

	def _hash_url(self, url: str) -> str:
		return hashlib.md5(url.encode()).hexdigest()

	def _get_cache_path(self, url: str) -> str:
		return os.path.join(CACHE_DIR, f"{self._hash_url(url)}.json")

	def map_url(self, url: str) -> List[str]:
		cache_path = self._get_cache_path(url)

		if os.path.exists(cache_path):
			with open(cache_path, 'r') as f:
				links = json.load(f)
				print(f"Loaded {len(links)} cached links for {url}")
				return links

		try:
			result = self.app.map_url(url)
			if getattr(result, 'success', False):
				links = result.links
				with open(cache_path, 'w') as f:
					json.dump(links, f, indent=2)
				print(f"Firecrawl found {len(links)} links for {url}")
				return links
			else:
				print(f"Firecrawl failed for {url}")
				return []
		except Exception as e:
			print(f"Firecrawl error for {url}: {e}")
			return []

	def filter_by_category(self, urls: List[str], category: str) -> List[str]:
		keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
		if not keywords:
			print(f"No keywords defined for category {category}")
			return []
		return [u for u in urls if any(k in u.lower() for k in keywords)]


def extract_main_html_content(html: str) -> str:
	soup = BeautifulSoup(html, "html.parser")

	for tag in soup(["script", "style", "noscript"]):
		tag.decompose()

	main = soup.find("main") or soup.find("article")
	if main:
		return main.get_text(separator="\n", strip=True)

	candidates = [
		div for div in soup.find_all("div")
		if len(div.get_text(strip=True)) > 200
		   and not any(c in " ".join(div.get("class", [])).lower() for c in ["nav", "header", "footer", "popup"])
	]
	if candidates:
		return max(candidates, key=lambda d: len(d.get_text(strip=True))).get_text(separator="\n", strip=True)

	return soup.get_text(separator="\n", strip=True)


async def crawl_and_select_best(urls: List[str], category: str) -> str:
	crawler_config = CrawlerRunConfig(
		deep_crawl_strategy=BFSDeepCrawlStrategy(max_depth=0),
		verbose=False
	)
	threshold = CATEGORY_THRESHOLDS.get(category.upper(), 150)

	best_text = ""
	best_wc = 0

	async with AsyncWebCrawler() as crawler:
		for url in urls:
			try:
				result = await asyncio.wait_for(crawler.arun(url, config=crawler_config), timeout=15)
				if not result:
					continue

				r = result[0]
				if r.html:
					text = extract_main_html_content(r.html)
					wc = len(text.split())
					print(f"{url} - {wc} words")

					if wc >= threshold:
						return text  # Return early

					if wc > best_wc:
						best_text, best_wc = text, wc

			except Exception as e:
				print(f"Error crawling {url}: {e}")

	return best_text or "No meaningful content found"


class GoogleSheetsManager:
	def __init__(self, credentials_file: str):
		scopes = ['https://www.googleapis.com/auth/spreadsheets']
		creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
		self.service = build('sheets', 'v4', credentials=creds)

	def extract_spreadsheet_id(self, sheet_url: str) -> str:
		import re
		pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
		match = re.search(pattern, sheet_url)
		if match:
			return match.group(1)
		raise ValueError("Invalid Google Sheet URL")

	def get_urls(self, spreadsheet_id: str) -> List[Dict]:
		range_name = f"{COLUMN_TO_READ_URL_FROM}2:{COLUMN_TO_READ_URL_FROM}"
		result = self.service.spreadsheets().values().get(spreadsheetId=spreadsheet_id, range=range_name).execute()
		values = result.get('values', [])
		return [(i + 2, row[0]) for i, row in enumerate(values) if row and row[0].strip()]

	def update_result(self, spreadsheet_id: str, row: int, content: str):
		col = COLUMN_TO_WRITE_URL_TO.get(COLUMN_TO_PROCESS.upper())
		if not col:
			print(f"Invalid COLUMN_TO_PROCESS: {COLUMN_TO_PROCESS}")
			return
		
		if len(content) > 50000:
			print(f"Truncating content from {len(content)} to 50000 characters.")
			content = content[:50000]

		range_str = f"{col}{row}"
		self.service.spreadsheets().values().update(
			spreadsheetId=spreadsheet_id,
			range=range_str,
			valueInputOption='RAW',
			body={'values': [[content]]}
		).execute()
		print(f"Updated row {row} in column {col}")


async def process_all_rows_firecrawl(sheet_url: str, credentials_file: str, firecrawl_api_key: str):
	sheet_mgr = GoogleSheetsManager(credentials_file)
	spreadsheet_id = sheet_mgr.extract_spreadsheet_id(sheet_url)
	urls = sheet_mgr.get_urls(spreadsheet_id)

	firecrawl = FirecrawlWrapper(api_key=firecrawl_api_key)

	for i, (row_num, main_url) in enumerate(urls):
		print(f"\nProcessing row {row_num}: {main_url}")
		sub_urls = firecrawl.map_url(main_url)
		await asyncio.sleep(6.5)

		filtered = firecrawl.filter_by_category(sub_urls, COLUMN_TO_PROCESS)
		filtered = sorted(filtered, key=lambda url: len(url))  # shortest first
		filtered = filtered[:10]



		if not filtered:
			content = "No relevant URLs found"
		else:
			content = await crawl_and_select_best(filtered, COLUMN_TO_PROCESS)

		sheet_mgr.update_result(spreadsheet_id, row_num, content)


In [2]:
FIRECRAWL_API="fc-29599096ac8b426dbf178180c53500ed"
CREDENTIALS_FILE = "data/url-to-email-445616-cebe4868914f.json"
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1QtKOB5ChRemg2_wJOxeZhn1qao1ZrRjVK8nExVzbxEI/edit?gid=2011509251#gid=2011509251" 

In [3]:
await process_all_rows_firecrawl(
	sheet_url=GOOGLE_SHEET_URL,
	credentials_file=CREDENTIALS_FILE,
	firecrawl_api_key=FIRECRAWL_API
)


Processing row 3: https://redkiteproject.com
Loaded 42 cached links for https://redkiteproject.com
Updated row 3 in column U

Processing row 4: https://lrscpa.com
Loaded 429 cached links for https://lrscpa.com


[INIT].... → Crawl4AI 0.6.3 

http://lrscpa.com/auto-body-shop - 145 words
Updated row 4 in column U

Processing row 5: https://ashtae.com
Loaded 262 cached links for https://ashtae.com


[INIT].... → Crawl4AI 0.6.3 

http://ashtae.com/pages/shop-by-style - 38 words
https://ashtae.com/pages/store-locator - 38 words
https://ashtae.com/products/saloncentric-in-store-class-mastering-loc-maintenance - 132 words
Updated row 5 in column U

Processing row 6: https://aithonsolutions.com
Loaded 27 cached links for https://aithonsolutions.com
Updated row 6 in column U

Processing row 7: https://hi-link.com
Loaded 102 cached links for https://hi-link.com
Updated row 7 in column U

Processing row 8: https://edaptschools.com
Loaded 18 cached links for https://edaptschools.com
Updated row 8 in column U

Processing row 9: https://globalmetalfinishing.com
Loaded 101 cached links for https://globalmetalfinishing.com


...

https://buffalosoldiersmuseum.org/press_releases/u-s-army-restores-honor-to-black-soldiers-hanged-in-jim-crow-era-south - 12 words
https://buffalosoldiersmuseum.org/press_releases/u-s-army-announces-3-step-plan-to-restore-honor-for-overturned-camp-logan-convictions - 13 words
Updated row 88 in column U

Processing row 89: https://atlasglinn.com
Loaded 39 cached links for https://atlasglinn.com
Updated row 89 in column U

Processing row 90: https://gansgans.com
Loaded 32 cached links for https://gansgans.com
Updated row 90 in column U

Processing row 91: https://beatbabel.com
Loaded 256 cached links for https://beatbabel.com
Updated row 91 in column U

Processing row 92: https://eagleresource.com
Loaded 6 cached links for https://eagleresource.com
Updated row 92 in column U

Processing row 93: https://curriculumredesign.org
Loaded 201 cached links for https://curriculumredesign.org
Updated row 93 in column U

Processing row 94: https://thinking-feet.com
Loaded 10 cached links for https:

[INIT].... → Crawl4AI 0.6.3 

https://www.ate1.org/preconference-workshop-1.html - 615 words
Updated row 96 in column U

Processing row 97: https://augs.org
Loaded 391 cached links for https://augs.org


[INIT].... → Crawl4AI 0.6.3 

https://www.augs.org/education-meetings/augs-at-home-workshop-series - 319 words
Updated row 97 in column U

Processing row 98: https://microburstlearning.com
Loaded 51 cached links for https://microburstlearning.com
Updated row 98 in column U

Processing row 99: https://tcgraleigh.com
Loaded 15 cached links for https://tcgraleigh.com
Updated row 99 in column U

Processing row 100: 97
Firecrawl error for 97: Unexpected error during map: Status code 400. Bad Request - [{'code': 'custom', 'message': 'URL must have a valid top-level domain or be a valid path', 'path': ['url']}]
Updated row 100 in column U

Processing row 101: 98.98
Firecrawl error for 98.98: Unexpected error during map: Status code 400. Bad Request - [{'code': 'custom', 'message': 'URL must have a valid top-level domain or be a valid path', 'path': ['url']}]
Updated row 101 in column U
